In [55]:
import pandas as pd
import numpy as np
import os
import calendar

In [56]:
# ==========================================
# 1. CONFIGURATION (Update these 2 numbers each month)
# ==========================================
REPORT_YEAR = 2026
REPORT_MONTH = 4  # Example: 4 = April, 5 = May, 6 = June

# ------------------------------------------
# DYNAMIC SETUP
# ------------------------------------------
month_name_full = calendar.month_name[REPORT_MONTH]
CURRENT_MONTH_COL = calendar.month_abbr[REPORT_MONTH]

# Ensure these match your actual file names
THIS_MONTH_FILE = f"This_Month_Data_Processed_Final_{month_name_full}_{REPORT_YEAR}.xlsx"
LAST_MONTH_FILE = "last_month.csv"

# THE SINGLE EXCEL FILE WITH MULTIPLE TABS
HISTORICAL_TRACKER_FILE = "six_month_metrics.xlsx"

TRACKER_SHEETS = {
    'Music Video': 'MV 6 Months',
    'Lyric Video': 'Lyric 6 Months',
    'Live Performance': 'Live 6 Months'
}

MASTER_DIR = f"Compliance_Reports_{month_name_full}_{REPORT_YEAR}"
LABEL_DIR = os.path.join(MASTER_DIR, "Individual_Label_Reports")
os.makedirs(LABEL_DIR, exist_ok=True)

print(f"--- Starting Video Compliance Generation for {month_name_full} {REPORT_YEAR} ---")

--- Starting Video Compliance Generation for April 2026 ---


In [57]:
# ==========================================
# 2. LOAD & PREP MAIN DATA
# ==========================================
if not os.path.exists(THIS_MONTH_FILE):
    raise FileNotFoundError(f"Could not find exact file: '{THIS_MONTH_FILE}'. Please check the name.")
if not os.path.exists(LAST_MONTH_FILE):
    raise FileNotFoundError(f"Could not find exact file: '{LAST_MONTH_FILE}'.")

print("Loading data... (This may take a moment for large Excel files)")
df_main = pd.read_excel(THIS_MONTH_FILE)
df_last = pd.read_csv(LAST_MONTH_FILE, encoding='utf-8', encoding_errors='replace', on_bad_lines='skip')

# Date conversions
df_main['NEW_DATE_dt'] = pd.to_datetime(df_main['NEW_DATE'], errors='coerce')
df_main['Upload Year'] = df_main['NEW_DATE_dt'].dt.year

# Fix casing duplicates dynamically
casing_map = df_main.groupby(df_main['report_name'].astype(str).str.lower())['report_name'].agg(lambda x: x.value_counts().index[0]).to_dict()
df_main['report_name'] = df_main['report_name'].astype(str).str.lower().map(casing_map)

# Combine ADA reports into 'ADA Int'
ada_mask = df_main['report_name'].str.contains('ADA', case=False, na=False) & ~df_main['report_name'].str.lower().isin(['ada us', 'ada uk'])
df_main.loc[ada_mask, 'report_name'] = 'ADA Int'

Loading data... (This may take a moment for large Excel files)


In [58]:
# ==========================================
# 3. FILTERING FOR CURRENT METRICS
# ==========================================
last_month_vids = set(df_last['VIDEO_ID'].dropna().unique())
mask_to_remove = (
    (df_main['dBOM'] == 'N') &
    (df_main['VIDEO_ID'].notna()) &
    (df_main['VIDEO_ID'] != 'n/a') &
    (~df_main['VIDEO_ID'].isin(last_month_vids)) &
    ((df_main['NEW_DATE_dt'].dt.month != REPORT_MONTH) | (df_main['Upload Year'] != REPORT_YEAR))
)
df_current = df_main[(df_main['Upload Year'] == REPORT_YEAR) & (~mask_to_remove)].copy()

In [59]:
# ==========================================
# 4. GENERATE GLOBAL PIVOTS & TRACKERS
# ==========================================
historical_trackers = {}
video_types = ['Music Video', 'Lyric Video', 'Live Performance']

updated_tracker_path = os.path.join(MASTER_DIR, f"Updated_6_Months_Metrics_{month_name_full}.xlsx")

with pd.ExcelWriter(updated_tracker_path, engine='xlsxwriter') as tracker_writer:

    for v_type in video_types:
        sheet_name = TRACKER_SHEETS.get(v_type)
        df_vtype = df_current[df_current['video type'].str.lower() == v_type.lower()].copy()
        safe_name = v_type.replace(' ', '_')
        
        dbomed_series = pd.Series(name=CURRENT_MONTH_COL, dtype=float)
        
        if not df_vtype.empty:
            pivot_df = pd.pivot_table(df_vtype, index='report_name', columns='Metric', aggfunc='size', fill_value=0)
            if 'Not dBOM' not in pivot_df.columns: pivot_df['Not dBOM'] = 0
                
            pivot_df.insert(0, 'Grand Total', pivot_df.sum(axis=1))
            pivot_df.loc['Grand Total'] = pivot_df.sum(axis=0)
            
            for col in [c for c in pivot_df.columns if c != 'Grand Total']:
                pivot_df[f"{col} (%)"] = ((pivot_df[col] / pivot_df['Grand Total']) * 100).fillna(0).round(2)
                
            pivot_df['% dBOMed'] = (((pivot_df['Grand Total'] - pivot_df['Not dBOM']) / pivot_df['Grand Total']) * 100).fillna(0).round(2)
            
            concatenated_col = pivot_df.index.astype(str) + " (" + pivot_df['Grand Total'].astype(int).astype(str) + ")"
            pivot_df.insert(0, 'Report Name (Total)', concatenated_col)
            
            pivot_df.to_csv(os.path.join(MASTER_DIR, f'Global_Metrics_Pivot_{safe_name}.csv'))
            
            dbomed_series = pivot_df['% dBOMed'].drop('Grand Total', errors='ignore')
            dbomed_series.name = CURRENT_MONTH_COL

        if os.path.exists(HISTORICAL_TRACKER_FILE):
            try:
                tracker_df = pd.read_excel(HISTORICAL_TRACKER_FILE, sheet_name=sheet_name)
                tracker_df = tracker_df.merge(dbomed_series, how='left', left_on='All PMV Data', right_index=True)
                tracker_df = tracker_df.fillna(0)  
                
                tracker_df.to_excel(tracker_writer, sheet_name=sheet_name, index=False)
                tracker_df.set_index('All PMV Data', inplace=True)
                historical_trackers[v_type] = tracker_df
            except ValueError:
                print(f"Warning: Could not find tab '{sheet_name}' in {HISTORICAL_TRACKER_FILE}.")
        else:
            print(f"Warning: '{HISTORICAL_TRACKER_FILE}' not found. Tracker generation skipped for {v_type}.")

In [60]:
# ==========================================
# 5. GENERATE INDIVIDUAL EXCEL REPORTS
# ==========================================
print("Generating individual label reports... Please wait.")
unique_reports = [name for name in df_main['report_name'].dropna().unique() if name != 'nan']
metrics_order = ['>3 days before street date', '2-3 days before street date', '1 day before street date', 
                 'Delivered on street date', 'Delivered after street date', 'Not dBOM']

valid_types = [v.lower() for v in video_types]

for report in unique_reports:
    file_path = os.path.join(LABEL_DIR, f"{report}_VideoComplianceReport_{month_name_full}_{REPORT_YEAR}.xlsx")
    
    df_report_all = df_main[df_main['report_name'] == report]
    df_report_current = df_current[df_current['report_name'] == report]
    
    valid_vtype_mask = df_report_all['video type'].astype(str).str.lower().isin(valid_types)
    
    missing_2026 = df_report_all[(df_report_all['dBOM'] == 'N') & (df_report_all['Upload Year'] == REPORT_YEAR) & valid_vtype_mask]
    missing_older = df_report_all[(df_report_all['dBOM'] == 'N') & (df_report_all['Upload Year'] < REPORT_YEAR) & valid_vtype_mask]
    
    with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
        workbook = writer.book
        summary_sheet = workbook.add_worksheet('Summary')
        
        # --- NEW EXCEL FORMATTING DEFINITIONS ---
        border_format = workbook.add_format({'border': 1})
        bold_border_format = workbook.add_format({'bold': True, 'border': 1, 'bg_color': '#F2F2F2'})
        
        # Updated: 0.00% ensures exactly two decimal places for the top metrics table
        pct_border_format = workbook.add_format({'num_format': '0.00%', 'border': 1})
        
        # Updated: 0.00 ensures exactly two decimal places for the 6-Month Tracker table
        tracker_val_format = workbook.add_format({'num_format': '0.00', 'border': 1})
        
        bold_format = workbook.add_format({'bold': True})
        
        # 1A. Write Current Metrics (Pivot)
        summary_sheet.write(0, 0, f"{REPORT_YEAR} Metrics", bold_format)
        row_idx = 1
        col_idx = 0
        summary_sheet.write(row_idx, col_idx, "Metric", bold_border_format)
        
        for v_type in video_types:
            col_idx += 1
            summary_sheet.write(row_idx, col_idx, v_type, bold_border_format)
            col_idx += 1
            summary_sheet.write(row_idx, col_idx, "%", bold_border_format)
        row_idx += 1
        
        for metric in metrics_order:
            summary_sheet.write(row_idx, 0, metric, border_format)
            col_idx = 1
            for v_type in video_types:
                count = len(df_report_current[(df_report_current['Metric'] == metric) & (df_report_current['video type'].str.lower() == v_type.lower())])
                total = len(df_report_current[df_report_current['video type'].str.lower() == v_type.lower()])
                pct = (count / total) if total > 0 else 0
                
                summary_sheet.write(row_idx, col_idx, count, border_format)
                summary_sheet.write(row_idx, col_idx + 1, pct, pct_border_format)
                col_idx += 2
            row_idx += 1
            
        row_idx += 2 
        
        # 1B. Write Historical 6-Month Tracker Data
        tracker_start_row = row_idx
        summary_sheet.write(row_idx, 0, "6 Month Tracker", bold_format)
        
        months = []
        if video_types[0] in historical_trackers:
            months = list(historical_trackers[video_types[0]].columns)
            for m_idx, month in enumerate(months):
                summary_sheet.write(row_idx, m_idx + 1, month, bold_border_format)
        
        row_idx += 1
        data_start_row = row_idx
        
        for v_type in video_types:
            summary_sheet.write(row_idx, 0, v_type, border_format)
            if v_type in historical_trackers and report in historical_trackers[v_type].index:
                history_data = historical_trackers[v_type].loc[report]
                for m_idx, month in enumerate(months):
                    val = history_data[month]
                    if pd.isna(val) or np.isinf(val): val = 0
                    # Applying the new two-decimal format here
                    summary_sheet.write(row_idx, m_idx + 1, val, tracker_val_format)
            else:
                for m_idx in range(len(months)):
                    summary_sheet.write(row_idx, m_idx + 1, 0, tracker_val_format)
            row_idx += 1
            
        # 1C. Create the Line Chart
        chart = workbook.add_chart({'type': 'line'})
        for i, v_type in enumerate(video_types):
            current_data_row = data_start_row + i
            chart.add_series({
                'name':       ['Summary', current_data_row, 0],
                'categories': ['Summary', tracker_start_row, 1, tracker_start_row, len(months)],
                'values':     ['Summary', current_data_row, 1, current_data_row, len(months)],
                'marker':     {'type': 'circle'}
            })
            
        chart.set_title({'name': f'6 Month % dBOMed - {report}'})
        chart.set_x_axis({'name': 'Month'})
        chart.set_y_axis({'name': '% dBOMed', 'max': 100}) 
        
        chart.set_style(10)
        
        summary_sheet.insert_chart(f'A{row_idx + 2}', chart, {'x_offset': 15, 'y_offset': 10})
        summary_sheet.set_column('A:A', 25) 
        
        # ----------- SHEET 2 & 3: MISSING VIDEOS -----------
        cols_to_export = ['report_name', 'marketing_label', 'ISRC', 'Workability', 'dBOM', 'VIDEO_ID', 'CHANNEL_ID', 'CHANNEL_DISPLAY_NAME', 'TITLE', 'video type', 'NEW_DATE', 'VIEW_COUNT']
        
        cols_to_export_2026 = [c for c in cols_to_export if c in missing_2026.columns]
        cols_to_export_older = [c for c in cols_to_export if c in missing_older.columns]
        
        missing_2026[cols_to_export_2026].to_excel(writer, sheet_name=f'Missing {REPORT_YEAR}', index=False)
        missing_older[cols_to_export_older].to_excel(writer, sheet_name='Missing Older', index=False)

print(f"\nSUCCESS! All outputs saved to: '{MASTER_DIR}'")

Generating individual label reports... Please wait.

SUCCESS! All outputs saved to: 'Compliance_Reports_April_2026'
